# Falsification of Specs using SpecForge SDK and Psy-Taliro

In this example notebook, we demonstrate how to perform falsification. The Falsification problem is as follows:
- Fixed System Model (i.e, a relation between input signals and output signals)
- Input: A Temporal Logic Specification on the input and output signals of the system
- Output: An Input Signal that causes the system to violate the specification

We use SpecForge to define, manage and monitor the specs; and we use [Psy-Taliro](https://psy-taliro.readthedocs.io/) to perform the falsification.

## Setup

In [ ]:
import logging
import math
from typing import Final

import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp


# Models of Automated Aircraft Maneuvers
# See https://github.com/cpslab-asu/aerobenchvvpython
from aerobench.examples.gcas.gcas_autopilot import GcasAutopilot
from aerobench.run_f16_sim import run_f16_sim

# Toolbox for search-based test generation for cyber-physical systems
# See https://github.com/cpslab-asu/psy-taliro
from staliro import TestOptions, Trace, staliro
from staliro.models import Blackbox, blackbox
from staliro.optimizers import DualAnnealing
from staliro.specifications import rtamt

TSPAN: Final[tuple[float, float]] = (0, 15)

from specforge_sdk import (
    SpecForgeClient,
    nested_encoding,
    flat_encoding,
    EXPORT_LILO,
    EXPORT_JSON,
    EXPORT_RTAMT,
    converters,
)
import pandas as pd
import json
import numpy as np
import os

Before running the following cell, make sure the SpecForge server is running.

In [ ]:
# Get the port from environment variable or use default
port = os.environ.get("SPECFORGE_PORT", "8080")

# Initialize the client
specforgeClient = SpecForgeClient(base_url="http://localhost:" + port)

# Check connection
if specforgeClient.health_check():
    print(f"✓ Connected to SpecForge API v{specforgeClient.version()}")
else:
    print("✗ Cannot connect to SpecForge API")
    print("Make sure the SpecForge server is running on http://localhost:" + port)

# Model Definition

Now, we define the F16 fighter jet model, following the example in the [Psy-Taliro documentation](https://psy-taliro.readthedocs.io/latest/examples.html).

It is defined as a _blackbox_ model, which means that the solver does not rely on the internal implementation of the model.

The main state variables of the model are roll, pitch, yaw, altitude, and a boolean indicating whether the aircraft is in standby mode.

In [ ]:
@blackbox()
def f16_model(inputs: Blackbox.Inputs) -> Trace[list[float]]:
    power = 9
    alpha = np.deg2rad(2.1215)
    beta = 0
    alt = 2330
    vel = 540
    phi = inputs.static["phi"]
    theta = inputs.static["theta"]
    psi = inputs.static["psi"]

    initial_state = [vel, alpha, beta, phi, theta, psi, 0, 0, 0, 0, 0, 0, alt, power]
    step = 1.0 / 30.0
    autopilot = GcasAutopilot(init_mode="roll", stdout=False)
    result = run_f16_sim(initial_state, TSPAN[1], autopilot, step, extended_states=True)
    states = np.vstack(
        (
            np.array([0 if x == "standby" else 1 for x in result["modes"]]),
            result["states"][:, 4],  # roll
            result["states"][:, 5],  # pitch
            result["states"][:, 6],  # yaw
            result["states"][:, 12],  # altitude
        )
    )

    return Trace(times=result["times"], states=np.transpose(states).tolist())

Use the Specforge SDK to translate the LILO specification to RTAMT format. The following formula says that the aircraft is always at a reasonable altitude.

In [ ]:
rtamt_formula = specforgeClient.export(
    system="f16",
    definition="afloat",
    export_type=EXPORT_RTAMT,
    return_string=True,  # Get the exported string
)
print("Exported RTAMT formula:", rtamt_formula)

## Setting up the Optimizer and Falsifier Options

Here we configure the details of the optimization process, including the choice of optimizer, number of iterations, and the mapping of the input to the system model.

In [ ]:
spec = rtamt.parse_dense(rtamt_formula, {"alt": 4})
optimizer = DualAnnealing()
options = TestOptions(
    runs=1,
    iterations=10,
    tspan=TSPAN,
    static_inputs={
        "phi": math.pi / 4 + np.array([-math.pi / 20, math.pi / 30]),
        "theta": -math.pi / 2 * 0.8 + np.array([0, math.pi / 20]),
        "psi": -math.pi / 4 + np.array([-math.pi / 8, math.pi / 8]),
    },
)

## Running the Falsifier

In [ ]:
logging.basicConfig(level=logging.DEBUG)

runs = staliro(f16_model, spec, optimizer, options)
run = runs[0]
min_cost_eval = min(run.evaluations, key=lambda e: e.cost)
min_cost_trace = min_cost_eval.extra.trace

Let's pick the trace with the minimum cost, and convert it into a pandas DataFrame

In [ ]:
dataframe = pd.DataFrame(
    {
        "time": [t for t in min_cost_trace.times],
        "mode": [bool(s[0]) for s in min_cost_trace.states],
        "roll": [s[1] for s in min_cost_trace.states],
        "pitch": [s[2] for s in min_cost_trace.states],
        "yaw": [s[3] for s in min_cost_trace.states],
        "alt": [s[4] for s in min_cost_trace.states],
    }
)
dataframe.head()

We can visualize the results by plotting the Signals

In [ ]:
figure = sp.make_subplots(rows=5, cols=1, shared_xaxes=True, x_title="Time (s)")

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["roll"], name="Roll"), row=1, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["pitch"], name="Pitch"), row=2, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["yaw"], name="Yaw"), row=3, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["alt"], name="Altitude"), row=4, col=1
)

figure.add_trace(
    go.Scatter(x=dataframe["time"], y=dataframe["mode"], name="Mode"), row=5, col=1
)

figure.update_yaxes(title_text="Roll", row=1, col=1)
figure.update_yaxes(title_text="Pitch", row=2, col=1)
figure.update_yaxes(title_text="Yaw", row=3, col=1)
figure.update_yaxes(title_text="Altitude", row=4, col=1)
figure.update_yaxes(title_text="Mode", row=5, col=1)

figure.update_layout(title_text="F16 Simulation Results")

figure.show()

Now, we can invoke the Specforge Monitor to visualize the trace. If you drag the slider around the 4.8 second mark, you will see that the altitude drops below 10, violating the specification.

In [ ]:
specforgeClient.monitor(system="f16", definition="afloat", data=dataframe)